### Advanced RAG Techniques!

In [24]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [25]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [26]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {
            "source": document.metadata.get("source"),
            "language": document.metadata.get("show_type", document.metadata.get("language")),
        }
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text, metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

In [27]:
from langchain_community.document_loaders import CSVLoader
from langchain_core import documents

def fetch_documents():
    # Define the path to your specific CSV file
    file_path = "knowledge-base/songs-dataset.csv"

    # Initialize the loader
    # Use 'metadata_columns' to automatically move the 'type' column into the metadata dict
    loader = CSVLoader(
        file_path=file_path,
        encoding='utf-8',
        metadata_columns=["language"]  # This extracts 'Movie' or 'TV Show' into metadata
    )

    # Load the data into document objects
    documents = loader.load()

    # Update the metadata key name to 'doc_type' as per your requirement
    for doc in documents:
        # 'type' is now in metadata because of the loader configuration above
        doc.metadata["show_type"] = doc.metadata.get("language")
        
        # Optional: Remove the original 'type' key if you only want 'doc_type'
        # del doc.metadata["type"]

    print(f"Loaded {len(documents)} rows.")
    print(f"Example Metadata: {documents[0].metadata}")

    return documents

In [28]:
documents = fetch_documents()

Loaded 101 rows.
Example Metadata: {'source': 'knowledge-base/songs-dataset.csv', 'row': 0, 'language': 'Telugu', 'show_type': 'Telugu'}


In [29]:
documents

[Document(metadata={'source': 'knowledge-base/songs-dataset.csv', 'row': 0, 'language': 'Telugu', 'show_type': 'Telugu'}, page_content='title: Samajavaragamana\nartist: Sid Sriram\nmovie: Ala Vaikunthapurramuloo\nemotion: Joy'),
 Document(metadata={'source': 'knowledge-base/songs-dataset.csv', 'row': 1, 'language': 'Telugu', 'show_type': 'Telugu'}, page_content='title: Butta Bomma\nartist: Armaan Malik\nmovie: Ala Vaikunthapurramuloo\nemotion: Trust'),
 Document(metadata={'source': 'knowledge-base/songs-dataset.csv', 'row': 2, 'language': 'Telugu', 'show_type': 'Telugu'}, page_content='title: Srivalli\nartist: Sid Sriram\nmovie: Pushpa\nemotion: Anticipation'),
 Document(metadata={'source': 'knowledge-base/songs-dataset.csv', 'row': 3, 'language': 'Telugu', 'show_type': 'Telugu'}, page_content='title: Oo Antava\nartist: Indravathi Chauhan\nmovie: Pushpa\nemotion: Joy'),
 Document(metadata={'source': 'knowledge-base/songs-dataset.csv', 'row': 4, 'language': 'Telugu', 'show_type': 'Telug

In [30]:
def make_prompt(document):
    text = document.page_content
    doc_type = document.metadata.get("show_type", document.metadata.get("language", "Unknown"))
    source = document.metadata.get("source", "Unknown")
    how_many = (len(text) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {doc_type}
The document has been retrieved from: {source}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{text}

Respond with the chunks.
"""

In [31]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [32]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: Telugu\nThe document has been retrieved from: knowledge-base/songs-dataset.csv\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 1 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\ntit

In [33]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [34]:
process_document(documents[0])

[Result(page_content="Song Details: Samajavaragamana\n\nThis chunk provides basic information about the song 'Samajavaragamana,' including its artist, movie, and the emotion it evokes.\n\ntitle: Samajavaragamana\nartist: Sid Sriram\nmovie: Ala Vaikunthapurramuloo\nemotion: Joy", metadata={'source': 'knowledge-base/songs-dataset.csv', 'language': 'Telugu'})]

In [35]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [36]:
chunks = create_chunks(documents)

100%|██████████| 101/101 [02:04<00:00,  1.23s/it]


In [37]:
print(len(chunks))

171


In [38]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [39]:
create_embeddings(chunks)

Vectorstore created with 171 documents


In [40]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['language'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['Telugu', 'Hindi', 'English', 'Marathi'].index(t)] for t in doc_types]

In [41]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [42]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Advanced RAG!

We will use these techniques:

- Reranking - reorder the rank results
- Query re-writing

In [43]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [44]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]


In [45]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [52]:
question = "Which artist sing a song Tera Yaar Hoon Main"
chunks = fetch_context_unranked(question)

In [53]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Introduction to...
Song Title and ...
Details about t...
Song Details: T...
Introduction to...
Song Details: T...
Introduction to...
Song Details: C...
Song Title and ...
Song Details an...
Additional Cont...
Song Details: '...
Agar Tum Saath ...
Raataan Lambiya...
Song Details: H...
Song Title and ...
Song Details: K...
Song Title and ...
Kesariya - Song...
Song Details: S...


In [54]:
reranked = rerank(question, chunks)

[1, 8, 2, 12, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20]


In [55]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Introduction to...
Song Details: C...
Song Title and ...
Song Details: '...
Details about t...
Song Details: T...
Introduction to...
Song Details: T...
Introduction to...
Song Title and ...
Song Details an...
Additional Cont...
Agar Tum Saath ...
Raataan Lambiya...
Song Details: H...
Song Title and ...
Song Details: K...
Song Title and ...
Kesariya - Song...
Song Details: S...


In [ ]:
question = "?Which artist sing a song Tera Yaar Hoon Main"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "Tera Yaar Hoon Main" in c.page_content.lower():
        print(index)


In [57]:
reranked = rerank(question, chunks)

[1, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [58]:
for index, c in enumerate(reranked):
    if "Tera Yaar Hoon Main" in c.page_content.lower():
        print(index)

In [59]:
reranked[0].page_content

"Introduction to 'Tera Yaar Hoon Main' from 'Sonu Ke Titu Ki Sweety'\n\nThis segment introduces the song 'Tera Yaar Hoon Main,' performed by Arijit Singh for the movie 'Sonu Ke Titu Ki Sweety,' evoking feelings of trust.\n\ntitle: Tera Yaar Hoon Main\nartist: Arijit Singh\nmovie: Sonu Ke Titu Ki Sweety\nemotion: Trust"

In [60]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [61]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [62]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [63]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [64]:
rewrite_query("Which artist sing a song Tera Yaar Hoon Main?", [])

'Who is the singer of "Tera Yaar Hoon Main"?'

In [65]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [66]:
answer_question("Which artist sing a song Tera Yaar Hoon Main?", [])

Who is the singer of "Tera Yaar Hoon Main"?
[1, 8, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


('The song "Tera Yaar Hoon Main" is sung by Arijit Singh.',
 [Result(page_content="Introduction to 'Tera Yaar Hoon Main' from 'Sonu Ke Titu Ki Sweety'\n\nThis segment introduces the song 'Tera Yaar Hoon Main,' performed by Arijit Singh for the movie 'Sonu Ke Titu Ki Sweety,' evoking feelings of trust.\n\ntitle: Tera Yaar Hoon Main\nartist: Arijit Singh\nmovie: Sonu Ke Titu Ki Sweety\nemotion: Trust", metadata={'source': 'knowledge-base/songs-dataset.csv', 'language': 'Hindi'}),
  Result(page_content="Song Details: Channa Mereya\n\nThis chunk provides essential details about the song 'Channa Mereya,' including its artist, the movie it belongs to, and the emotion it conveys.\n\ntitle: Channa Mereya\nartist: Arijit Singh\nmovie: Ae Dil Hai Mushkil\nemotion: Sadness", metadata={'language': 'Hindi', 'source': 'knowledge-base/songs-dataset.csv'}),
  Result(page_content="Song Title and Artist\n\nThe song is titled 'Channa Mereya' and is sung by Arijit Singh, a popular Indian playback singer.\

In [67]:
answer_question("Which artist sing a song Sun Saathiya?", [])

Who is the singer of the song Sun Saathiya?
[1, 4, 5, 6, 8, 11, 13, 14, 16, 18, 19, 20, 2, 3, 7, 9, 10, 12, 15, 17]


('The artist who sings the song "Sun Saathiya" is Priya Saraiya.',
 [Result(page_content="Sun Saathiya - Song Details\n\nThis chunk provides detailed information about the song 'Sun Saathiya,' including its title, artist, movie, and emotion conveyed.\n\ntitle: Sun Saathiya\nartist: Priya Saraiya\nmovie: ABCD 2\nemotion: Joy", metadata={'source': 'knowledge-base/songs-dataset.csv', 'language': 'Hindi'}),
  Result(page_content="Introduction to 'Tera Yaar Hoon Main' from 'Sonu Ke Titu Ki Sweety'\n\nThis segment introduces the song 'Tera Yaar Hoon Main,' performed by Arijit Singh for the movie 'Sonu Ke Titu Ki Sweety,' evoking feelings of trust.\n\ntitle: Tera Yaar Hoon Main\nartist: Arijit Singh\nmovie: Sonu Ke Titu Ki Sweety\nemotion: Trust", metadata={'language': 'Hindi', 'source': 'knowledge-base/songs-dataset.csv'}),
  Result(page_content="Song Title and Artist\n\nThis chunk provides the basic details about the song 'Tum Hi Ho', including the artist, movie, and emotion conveyed.\n\nti